# Cross-validation calibration

This notebook generates out-of-fold (OOF) calibration inputs before fitting the final learner on all training observations. It covers ICP, CQR, CPS, and binary classification.

In [2]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from quantile_forest import RandomForestQuantileRegressor

from tinyconformal.core.calibration import CrossValidationCalibration
from tinyconformal.classifier import (
    BinaryClassConditionalConformalClassifier,
    BinaryMarginalConformalClassifier,
)
from tinyconformal.distribution import ContinuousCrossConformalPredictiveSystem
from tinyconformal.regressor import (
    ConformalizedQuantileRegressor,
    ConformalizedRegressor,
)

In [3]:
X, y = make_regression(n_samples=800, n_features=10, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## ICP from OOF absolute-residual scores

In [4]:
point_learner = RandomForestRegressor(n_estimators=100, random_state=42)
icp_scores = CrossValidationCalibration.icp_scores(
    point_learner, X_train, y_train, cv=5, n_jobs=-1
)
point_learner.fit(X_train, y_train)
icp = ConformalizedRegressor(point_learner).fit_from_scores(icp_scores)
icp_intervals = icp.predict_interval(X_test)
icp_intervals[:3]

array([[-319.43682325,   85.54091268],
       [-151.26230521,  253.71543072],
       [-107.11916543,  297.8585705 ]])

## CQR from OOF quantile scores

In [5]:
quantile_learner = RandomForestQuantileRegressor(
    n_estimators=100, default_quantiles=[0.025, 0.975], random_state=42
)
cqr_scores = CrossValidationCalibration.cqr_scores(
    quantile_learner, X_train, y_train, cv=5, n_jobs=-1
)
quantile_learner.fit(X_train, y_train)
cqr = ConformalizedQuantileRegressor(quantile_learner).fit_from_scores(cqr_scores)
cqr.predict_interval(X_test)[:3]

array([[-306.8715373 ,   18.87487774],
       [-151.93457678,  271.21567149],
       [-124.12425091,  361.05027547]])

## Cross-fitted CPS with locally scaled residuals

In [6]:
cps_learner = RandomForestRegressor(n_estimators=100, random_state=42)
dispersion_learner = RandomForestRegressor(n_estimators=100, random_state=42)
cps = ContinuousCrossConformalPredictiveSystem(
    cps_learner, dispersion_learner, cv=5, n_jobs=-1
).fit(X_train, y_train)
distribution = cps.predict_distribution(X_test)
distribution.interval(0.9)[:3]

array([[-276.57373572,   56.9070027 ],
       [-108.39921768,  225.08152074],
       [ -64.2560779 ,  269.22466052]])

## Classification from OOF probabilities

In [7]:
Xc, yc = make_classification(n_samples=800, n_features=10, random_state=42)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, stratify=yc, random_state=42
)
classifier_learner = RandomForestClassifier(n_estimators=100, random_state=42)
oof_probabilities = CrossValidationCalibration.classification_probabilities(
    classifier_learner, Xc_train, yc_train, cv=5, n_jobs=-1
)
classifier_learner.fit(Xc_train, yc_train)
marginal = BinaryMarginalConformalClassifier(
    classifier_learner
).fit_from_probabilities(oof_probabilities, yc_train)
class_conditional = BinaryClassConditionalConformalClassifier(
    classifier_learner
).fit_from_probabilities(oof_probabilities, yc_train)
marginal.predict_set(Xc_test)[:5], class_conditional.predict_set(Xc_test)[:5]

(array([[0, 1],
        [1, 0],
        [1, 0],
        [0, 1],
        [0, 1]]),
 array([[0, 1],
        [1, 0],
        [1, 0],
        [0, 1],
        [0, 1]]))